In [ ]:
# 🛠️ Setup: install deps + fetch the shared helper on Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformer_lens datasets transformers plotly
    # transformer_lens pulls a newer torch; Colab's torchaudio was built against the
    # old torch and fails to load (undefined symbol: torch_library_impl). transformers
    # imports it lazily and crashes, though nothing here uses audio. Remove it.
    !pip uninstall -y -q torchaudio
    !wget -q https://raw.githubusercontent.com/barmag/fanous-llm-lens/main/notebooks/education/tiny.py
import tiny
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

torch.manual_seed(42)
print("device:", tiny.device())

<div dir="rtl">

# المرحلة ٢ب: كذا رأس — كذا علاقة في نفس الوقت

في المرحلة ٢أ كان عندنا **رأس انتباه واحد**: بيمسك علاقة واحدة بين التوكنز. بس اللغة فيها كذا علاقة شغّالة مع بعض (مين بيعمل الفعل؟ الصفة بتوصف مين؟ أداة التعريف بتاعة مين؟).

النهارده بنضيف **كذا رأس** (multiple heads) في نفس الطبقة. كل رأس بيتعلم يبص لحاجة مختلفة — فالموديل يقدر يمسك **كذا علاقة في نفس الوقت**، بالتوازي.

**حدود التجربة:** برضه موديل صغير على بيانات قليلة؛ التوقعات هتفضل ضعيفة. اللي يهمنا: نشوف الرؤوس **بتتخصص** في حاجات مختلفة.

</div>

<div dir="ltr">

# Stage 2b: Multiple heads — several relations at once

In 2a we had **one attention head**: it captures one relation between tokens. But language has several relations going at once (who does the verb? what does the adjective describe? whose is the article?).

Today we add **multiple heads** in the same layer. Each head learns to look at something different — so the model captures **several relations in parallel**.

**Limits:** still a tiny model on little data, so predictions stay weak. What matters: watching the heads **specialise** on different things.

</div>

<div dir="rtl">

## ١. البيانات والمُجزّئ · Data & tokenizer

نفس مصادر ومُجزّئ المرحلة ٢أ: نصوص فصحى + مصري، ومُجزّئ **mGPT** مُدرّب (بيفصل «ال») مع تصغير القاموس عشان الموديل يفضل قابل للتدريب. (المقارنة الكاملة بين mGPT و MARBERT في نوت بوك ٢أ.)

</div>

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import re

MAX_CHARS = globals().get("MAX_CHARS", 500_000)

print("Fetching MSA from Wikipedia and Masri from Tweets...")
msa_stream = load_dataset("wikimedia/wikipedia", "20231101.ar", split="train", streaming=True)
tweets_ds = load_dataset("amgadhasan/arabic_tweets_dialects", split="train")
eg_tweets = tweets_ds.filter(lambda x: x["dialect"] == "EG")

def clean(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[a-zA-Z0-9_@]+", "", text)
    return re.sub(r"[^\s\u0621-\u064A]", "", text)

def collect(rows, key, max_chars):
    out = ""
    for r in rows:
        out += clean(r[key]) + " "
        if len(out) >= max_chars:
            break
    return out[:max_chars]

msa_text = collect(msa_stream, "text", MAX_CHARS)
masri_text = collect(eg_tweets, "text", MAX_CHARS)

mgpt = AutoTokenizer.from_pretrained("ai-forever/mGPT")
encode, corpus_ids, id_to_str, VOCAB = tiny.make_compact_encoder(mgpt, [msa_text, masri_text])
all_ids = corpus_ids[0] + corpus_ids[1]
print(f"compact vocab: {VOCAB} subwords  |  tokens: {len(all_ids)}")  # ← (n_ids,)

<div dir="rtl">

## ٢. نبني الموديل وندرّبه · Build & train

موديل بطبقة واحدة (`n_layers=1`) بس بـ**٤ رؤوس** (`n_heads=4`). الفرق الوحيد عن المرحلة ٢أ هو عدد الرؤوس.

</div>

In [ ]:
N_CTX = globals().get("N_CTX", 64)
N_EPOCHS = globals().get("N_EPOCHS", 200)
N_HEADS = globals().get("N_HEADS", 4)

batches = tiny.make_natural_batches(all_ids, n_ctx=N_CTX)
print("batches:", tuple(batches.shape))  # ← (N, n_ctx)

model = tiny.make_tiny_model(n_layers=1, n_heads=N_HEADS, d_vocab=VOCAB, n_ctx=N_CTX)
losses = tiny.train(model, batches, n_epochs=N_EPOCHS, log_every=20)
print("loss:", round(losses[0], 3), "->", round(losses[-1], 3))  # ← scalar

<div dir="rtl">

## ٣. الرؤوس بتتخصص · The heads specialise

لكل رأس خريطة انتباه خاصة بيه. هنعرضهم جنب بعض على نفس الجملة. دوّر على الفرق: رأس ممكن يبص للكلمة اللي **قبل** كل توكن مباشرة، رأس تاني للكلمة الأولى، رأس تالت يوزّع انتباهه بالتساوي... كل واحد بيمسك علاقة مختلفة. (RTL: أول توكن على اليمين.)

</div>

In [ ]:
prompt = "القطة الكبيرة بتاكل السمك"
comp = encode(prompt)  # compact ids for the model
ids = torch.tensor([comp]).to(tiny.device())
str_toks = [id_to_str[i] for i in comp]

_, cache = model.run_with_cache(ids)
patterns = cache["pattern", 0][0]  # ← (n_heads, seq, seq)

fig = make_subplots(rows=1, cols=N_HEADS,
                    subplot_titles=[f"head {h}" for h in range(N_HEADS)])
for h in range(N_HEADS):
    p = patterns[h].detach().cpu().numpy()
    fig.add_trace(go.Heatmap(z=p, x=str_toks, y=str_toks, colorscale="Blues",
                             showscale=False), row=1, col=h + 1)
    fig.update_xaxes(autorange="reversed", row=1, col=h + 1)
    fig.update_yaxes(autorange="reversed", row=1, col=h + 1)
fig.update_layout(height=320, width=240 * N_HEADS,
                  title_text="نمط الانتباه لكل رأس · attention pattern per head")
fig.show()

<div dir="rtl">

## ٤. السياق بيغيّر التوقع — مع رؤوس أكتر · Context still steers the prediction

زي المرحلة ٢أ، بنرسم **شجرة التوقعات** على نفس السياقين (نفس الكلمة الأخيرة، بداية مختلفة). الموديل لسه بيستعمل السياق — بس دلوقتي عنده أكتر من رأس يجمع منهم معلومات. قارن بشجرة ٢أ: مع التدريب الكامل المفروض التوقعات تبقى أحسن شوية. (أحمر = `[UNK]` أو احتمال `< 0.10`.)

</div>

In [ ]:
LOW_PROB = 0.10

def next_topk(context_ids, k):
    ids = torch.tensor([context_ids]).to(tiny.device())
    probs = torch.softmax(model(ids)[0, -1], dim=-1)
    vals, idx = torch.topk(probs, k)
    return [(int(i), float(v)) for v, i in zip(vals, idx)]

def build_prediction_tree(context_text, max_depth=2, top_k=2):
    edges, nodes = [], {"root": (context_text, True)}
    frontier, n = [("root", encode(context_text), 0)], 0
    while frontier:
        pkey, ctx, depth = frontier.pop(0)
        if depth >= max_depth:
            continue
        for tid, p in next_topk(ctx, top_k):
            n += 1
            label = id_to_str.get(tid, "[UNK]")
            sensible = (label != "[UNK]") and (p >= LOW_PROB)
            ckey = f"{depth}:{n}:{label}"
            nodes[ckey] = (label, sensible)
            edges.append((pkey, ckey, p, depth, sensible))
            frontier.append((ckey, ctx + [tid], depth + 1))
    return edges, nodes

def tree_layout(edges):
    depth_of = {"root": 0}
    for _pk, ck, _p, d, _s in edges:
        depth_of[ck] = d + 1
    levels = {}
    for k, d in depth_of.items():
        levels.setdefault(d, []).append(k)
    return {k: (d, (len(ks) - 1) / 2 - i)
            for d, ks in levels.items() for i, k in enumerate(ks)}

def add_tree(fig, col, edges, nodes):
    sfx = "" if col == 1 else str(col)
    xref, yref = f"x{sfx}", f"y{sfx}"
    pos = tree_layout(edges)
    for pk, ck, p, _d, s in edges:
        (x0, y0), (x1, y1) = pos[pk], pos[ck]
        color = "#c0392b" if not s else "#2b6cb0"
        fig.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref=xref, yref=yref, axref=xref,
                           ayref=yref, showarrow=True, arrowhead=3, arrowwidth=1.5 + p * 7.5,
                           arrowcolor=color, standoff=18, startstandoff=24, text="")
        fig.add_annotation(x=(x0 + x1) / 2, y=(y0 + y1) / 2, xref=xref, yref=yref,
                           showarrow=False, text=f"{p:.2f}", font=dict(size=10, color="#333"),
                           bgcolor="rgba(255,255,255,0.75)")
    keys = list(pos.keys())
    fig.add_trace(go.Scatter(
        x=[pos[k][0] for k in keys], y=[pos[k][1] for k in keys], mode="markers+text",
        text=[nodes[k][0] for k in keys], textposition="middle center",
        textfont=dict(size=13, color="#111"),
        marker=dict(color=["#f6c350" if k == "root" else ("#f6c0bb" if not nodes[k][1] else "#d9d9d9")
                           for k in keys],
                    size=[40 if k == "root" else 30 for k in keys], line=dict(width=1.5, color="#333")),
        hoverinfo="text", showlegend=False), row=1, col=col)
    fig.update_xaxes(visible=False, row=1, col=col)
    fig.update_yaxes(visible=False, row=1, col=col)

CONTEXTS = ["القطة بتاكل السمك", "الولد بياكل السمك"]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=[f"السياق: {c}" for c in CONTEXTS])
for i, c in enumerate(CONTEXTS):
    e, n = build_prediction_tree(c, max_depth=2, top_k=2)
    add_tree(fig, i + 1, e, n)
fig.update_layout(height=460, margin=dict(l=20, r=20, t=80, b=20),
                  title_text="٤ رؤوس: السياق بيغيّر التوقع · multi-head — context changes the prediction")
fig.show()

<div dir="rtl">

## ٥. الخلاصة والخطوة الجاية · Recap & handoff

- ضفنا **كذا رأس** في نفس الطبقة؛ كل رأس بيتخصص في علاقة مختلفة — الموديل بيمسك كذا علاقة بالتوازي.
- شفنا **خريطة لكل رأس** جنب بعض، والفرق بينهم.
- أعدنا **شجرة التوقعات** على نفس السياقين — السياق لسه بيغيّر التوقع، بمعلومات من كذا رأس.

**المرحلة الجاية (٢ج):** نضيف **طبقة تانية**. لحد دلوقتي كل الرؤوس في طبقة واحدة وبتشتغل بالتوازي. العمق بيفتح حاجة جديدة: **التركيب** (composition) — رأس يستعمل ناتج رأس تاني — واللي منه بيطلع **رأس الاستقراء** (induction head).

</div>

<div dir="ltr">

## 5. Recap & handoff

- We added **multiple heads** in one layer; each specialises on a different relation — several relations captured in parallel.
- We saw **one heatmap per head** side by side, and how they differ.
- We reused the **prediction tree** on the same two contexts — context still steers the prediction, now pooling several heads.

**Next (2c):** add a **second layer**. So far all heads sit in one layer working in parallel. Depth unlocks something new: **composition** — one head using another head's output — which gives rise to the **induction head**.

</div>